In [6]:
from __future__ import annotations
import hydra
import sys
import numpy as np
import time
import json

from omegaconf import DictConfig
from hydra.utils import to_absolute_path

from pathlib import Path
from typing import Dict, Any

import numpy as np
import yaml

from env.load_map import load_map
from persona.load_personas import load_agent_configs
from env.grid import MultiHumanGridEnv
from env.constants import Action, DIR_TO_VEC 

# this might be temporary map until we move this high-level to the other cognitive models
from persona.cognitive.plan import get_accessible_locations, high_level_planner, astar, get_plan_for_time, normalize_command_for_planner, get_agent_tile
from env.constants import SemanticMap
from persona.cognitive.perceive import describe_perception, is_intersection, valid_move_actions, action_names

#---------------------------------------------------------------
## activate after env integration for hazards. remove the upper line
#from persona.cognitive.perceive import describe_perception, get_local_hazards
#---------------------------------------------------------------

from persona.prompt.gpt_structure import test_chat_completion, LLMConversation, llm_decide_intent, llm_decide_local_direction

import os
import datetime

map = load_map('/Users/waldburger/Desktop/cs294_286_final_project/grid_gen/configs/maps/urbanWorld.json')

/Users/waldburger/opt/anaconda3/envs/eLW085/lib/python3.9/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [7]:
np.asarray(map.access_grid).shape

(23, 31)

In [ ]:
for r in map.regions:
    if r['type'] == 'fire':
        fire_loc = (r['x'],r['y'])

[{'name': 'B1',
  'group_ids': ['row_top', 'col_west'],
  'type': 'block',
  'access_code': 0,
  'render': 'block',
  'shape': 'rect',
  'x': 2,
  'y': 2,
  'w': 5,
  'h': 5},
 {'name': 'B2',
  'group_ids': ['row_top', 'col_cw'],
  'type': 'block',
  'access_code': 0,
  'render': 'block',
  'shape': 'rect',
  'x': 9,
  'y': 2,
  'w': 5,
  'h': 5},
 {'name': 'B3',
  'group_ids': ['row_top', 'col_ce'],
  'type': 'block',
  'access_code': 0,
  'render': 'block',
  'shape': 'rect',
  'x': 16,
  'y': 2,
  'w': 5,
  'h': 5},
 {'name': 'B4',
  'group_ids': ['row_top', 'col_east'],
  'type': 'block',
  'access_code': 0,
  'render': 'block',
  'shape': 'rect',
  'x': 23,
  'y': 2,
  'w': 5,
  'h': 5},
 {'name': 'B5',
  'group_ids': ['row_mid', 'col_west'],
  'type': 'block',
  'access_code': 0,
  'render': 'block',
  'shape': 'rect',
  'x': 2,
  'y': 9,
  'w': 5,
  'h': 5},
 {'name': 'B7',
  'group_ids': ['row_mid', 'col_ce'],
  'type': 'block',
  'access_code': 0,
  'render': 'block',
  'shape

In [7]:
!python main.py

/Users/waldburger/opt/anaconda3/envs/eLW085/lib/python3.9/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
[Env] Resetting...
Initial state:
[VALID LOCATIONS] ['Home_A', 'Home_B', 'Park', 'Workplace_A', 'fire', 'park']
[2025-11-24 16:40:37,319][httpx][INFO] - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[OpenAI] Response: OpenAI comms successful.
[2025-11-24 16:40:43,566][httpx][INFO] - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[LLM INIT SUMMARY]
 Summary of agents' demographics aligned with priors:

- Isabella Rodriguez:
  - Age: 34 → falls into "25-34" age bucket.
  - Gender: female.
  - According to priors for "25-34" women:
 